In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "cacchione2009gravity")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "cacchione_2009_SPSS-table_raw-data_repaired.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)

df['study_id']="cacchione2009gravity"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

df.rename(columns={"subject": "ape",
    "species":"species_original"}, inplace=True)


In [3]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [4]:
# df['condition'].unique()


In [5]:
code_list=["condition"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == 'horizontal baseline':
            entry = "baseline"
        elif entry =='horizontal switch':
            entry = "baseline"
        elif entry =='vertical baseline':
            entry = "baseline"
        elif entry =='vertical switch':
            entry = "baseline"
        elif entry =='vertical test far':
            entry = "1"
        elif entry =='vertical test near':
            entry = "1"
        elif entry =='horizontal test far':
            entry = "2"
        elif entry =='horizontal test near':
            entry = "2"
        elif entry =='horizontal test far opposite c.':
            entry = "2"
        elif entry =='vertical test slight alignment far':
            entry = "3"
        elif entry =='vertical test great alignment far':
            entry = "3"
        elif entry =='vertical test great alignment far opposite c.':
            entry = "3"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': 'experiment'})

In [6]:
df.rename(columns={"ape": "participant"}, inplace=True)

df.replace(' ', '_', inplace=True, regex=True)
# df.columns
df.dropna(subset=['sex'], inplace=True)

In [7]:
df=df[['study_id', 'experiment','participant', 'sex','species','session', 'trial', 'condition',
       'response', 'correct']]

In [8]:
for index in range(1,4):
    exp = df[df['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'cacchione2009gravity_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'cacchione2009gravity_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

In [9]:
exp1 = df[df['experiment'] == 'baseline']
exp1 = exp1.dropna(axis=1, how='all')

comp_out_path_stand = os.path.join(out_pathway, 'cacchione2009gravity_baseline_standardized.csv')
exp1.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =exp1.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
cacchione2009gravity_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'cacchione2009gravity_baseline_glossary.csv')
cacchione2009gravity_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)